# 4) Adapter OOD Recalibration (Colab)

Use this notebook to find an existing adapter export, recalibrate its OOD state, and overwrite the Drive copy in place so it is ready to use immediately.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import os
import sys
from pathlib import Path

try:
    from scripts.colab_repo_bootstrap import (
        install_colab_requirements,
        mount_drive_if_available,
        resolve_repo_root,
        running_in_colab,
    )
except ModuleNotFoundError:
    clone_target = Path('/content/bitirmeprojesi')
    if not (clone_target / 'scripts').exists():
        import subprocess
        repo_url = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
        subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(clone_target)], check=True)
    os.chdir(clone_target)
    sys.path.insert(0, str(clone_target))
    from scripts.colab_repo_bootstrap import (
        install_colab_requirements,
        mount_drive_if_available,
        resolve_repo_root,
        running_in_colab,
    )

IN_COLAB = running_in_colab()
if IN_COLAB:
    mount_drive_if_available(force_remount=False)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

install_colab_requirements(ROOT / 'colab_notebooks' / 'requirements_colab.txt', IN_COLAB)

from scripts.recalibrate_adapter_ood import run_recalibration
from scripts.colab_adapter_smoke_test import discover_adapter_candidates

print(f'[SETUP] root={ROOT}')

In [ ]:
# Edit these values, then run the remaining cells top-to-bottom.

# Search root for adapters. This can be a telemetry root or any parent folder.
ADAPTER_SEARCH_ROOT = '/content/drive/MyDrive/aads_ulora/telemetry'

# Runtime dataset root containing train/val/test splits for the crop.
DATASET_ROOT = '/content/drive/MyDrive/aads_ulora/runtime_dataset'

# Crop name for the adapter you want to recalibrate.
CROP_NAME = 'tomato'

# Device used for recalibration.
DEVICE = 'cuda'

# Optional: set to a specific folder if you do not want in-place overwrite.
OUTPUT_DIR = None

In [ ]:
candidates = discover_adapter_candidates([ADAPTER_SEARCH_ROOT], crop_name=CROP_NAME)
print(f'Found {len(candidates)} candidate adapter(s).')
for idx, candidate in enumerate(candidates[:20], start=1):
    print(f"[{idx}] crop={candidate.get('crop_name')} calibrated_v={candidate.get('ood_calibration_version')} path={candidate.get('adapter_dir')}")

if not candidates:
    raise RuntimeError('No adapter bundles were found under ADAPTER_SEARCH_ROOT.')

In [ ]:
result = run_recalibration(
    adapter_ref=ADAPTER_SEARCH_ROOT,
    data_dir=DATASET_ROOT,
    crop_name=CROP_NAME,
    output_dir=OUTPUT_DIR,
    config_env='colab',
    device=DEVICE,
)
print(json.dumps(result, indent=2))

In [ ]:
meta_path = Path(result['adapter_output_dir']) / 'adapter_meta.json'
meta = json.loads(meta_path.read_text(encoding='utf-8'))

ood_state = dict(meta.get('ood_state', {}))
class_stats = dict(ood_state.get('class_stats', {}))
summary = {
    'adapter_meta_path': str(meta_path),
    'ood_calibration_version': int(dict(meta.get('ood_calibration', {})).get('version', 0)),
    'ood_state_calibration_version': int(ood_state.get('calibration_version', 0)),
    'class_stats_count': len(class_stats),
    'radial_beta': ood_state.get('radial_beta'),
    'conformal_qhat': ood_state.get('conformal_qhat'),
}
print(json.dumps(summary, indent=2))

if summary['class_stats_count'] <= 0:
    raise RuntimeError('Recalibration finished, but class_stats is still empty.')

print('Adapter is recalibrated and ready to use from Drive.')